In [0]:
from pyspark.sql import functions as F
from collections import Counter

def build_bronze(kind, columns):
    paths = sorted(
        f.path
        for f in project_files
        if f.name.startswith(f"sample_{kind}_")
    )

    lines = (
        spark.read.text(paths)
        .select(
            F.col("value").alias("raw_line"),
            F.col("_metadata.file_name").alias("source_file"),
            F.col("_metadata.file_path").alias("source_path")
        )
        .withColumn(
            "_fields",
            F.split(F.col("raw_line"), r"\|", -1)
        )
    )

    # Stop before saving if any row has an unexpected structure.
    invalid = lines.filter(
        (F.size("_fields") != len(columns))
        | (F.trim(F.col("raw_line")) == "")
    )

    assert invalid.limit(1).count() == 0, \
        f"Invalid row structure found in {kind} files."

    return lines.select(
        *[
            F.col("_fields").getItem(i).alias(name)
            for i, name in enumerate(columns)
        ],
        "raw_line",
        "source_file",
        "source_path",
        F.regexp_extract(
            F.col("source_file"), r"_(\d{4})\.txt$", 1
        ).cast("int").alias("source_vintage"),
        F.current_timestamp().alias("ingested_at")
    )

In [0]:
display(spark.sql("""
    SELECT
        'origination' AS table_name,
        COUNT(*) AS row_count,
        COUNT(DISTINCT source_file) AS source_files
    FROM workspace.mortgage_risk.bronze_origination

    UNION ALL

    SELECT
        'performance' AS table_name,
        COUNT(*) AS row_count,
        COUNT(DISTINCT source_file) AS source_files
    FROM workspace.mortgage_risk.bronze_performance
"""))

table_name,row_count,source_files
origination,500000,10
performance,20079736,10
